# 02 · Train, evaluate and register the litigation model

**Question:** at intake, how likely is a workers' comp claim to get an attorney / representative?

Steps
1. Pull features from `WC_ML.ANALYTICS.CLAIM_FEATURES` (the same view used for scoring).
2. Split by time: train 2017–2020, validate 2021, test 2022.
3. Fit a logistic-regression baseline and an XGBoost model.
4. Evaluate on the test year.
5. Log the XGBoost pipeline to the **Snowflake Model Registry** and make it the default version.

**Packages:** in the notebook's *Packages* menu add `snowflake-ml-python`, `scikit-learn` and `xgboost`.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
import pandas as pd

session = get_active_session()
session.use_warehouse("WC_ML_WH")
pd.set_option("display.max_columns", 50)

def q(sql: str) -> pd.DataFrame:
    """Run SQL in Snowflake and return a pandas DataFrame."""
    return session.sql(sql).to_pandas()

In [ ]:
FEATURE_VIEW = "WC_ML.ANALYTICS.CLAIM_FEATURES"
MODEL_NAME = "WC_LITIGATION_XGB"
LABEL = "IS_LITIGATED"

# Order matters: the registered model's signature (and the SQL in deploy/) follows this order.
NUMERIC = ["AGE_AT_INJURY", "REPORT_LAG_DAYS", "ACCIDENT_MONTH", "ACCIDENT_DOW"]
CATEGORICAL = ["GENDER", "CARRIER_TYPE", "DISTRICT_NAME", "COUNTY_OF_INJURY", "MEDICAL_FEE_REGION",
               "INDUSTRY_CODE", "WCIO_CAUSE_CODE", "WCIO_NATURE_CODE", "WCIO_BODY_CODE", "COVID_19_INDICATOR"]
FEATURES = NUMERIC + CATEGORICAL

TRAIN_YEARS = (2017, 2020)
VALID_YEAR = 2021
TEST_YEAR = 2022
MAX_TRAIN_ROWS = 600_000   # keeps memory and time modest on an X-Small warehouse
RANDOM_STATE = 42

## 1. Load and split by time

In [ ]:
df = (
    session.table(FEATURE_VIEW)
    .filter(F.col("ASSEMBLY_YEAR").between(TRAIN_YEARS[0], TEST_YEAR))
    .select("CLAIM_IDENTIFIER", "ASSEMBLY_YEAR", LABEL, *FEATURES)
    .to_pandas()
)
df[NUMERIC] = df[NUMERIC].astype("float64")
df[CATEGORICAL] = df[CATEGORICAL].astype(str)
df[LABEL] = df[LABEL].astype(int)

train_df = df[df.ASSEMBLY_YEAR.between(*TRAIN_YEARS)]
if len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)
valid_df = df[df.ASSEMBLY_YEAR == VALID_YEAR]
test_df = df[df.ASSEMBLY_YEAR == TEST_YEAR]

X_train, y_train = train_df[FEATURES], train_df[LABEL]
X_valid, y_valid = valid_df[FEATURES], valid_df[LABEL]
X_test, y_test = test_df[FEATURES], test_df[LABEL]

pd.DataFrame({
    "rows": [len(train_df), len(valid_df), len(test_df)],
    "litigation_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "valid", "test"]).round(4)

## 2. Baseline: logistic regression

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


def evaluate(name, y, p):
    return {"model": name, "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "brier": brier_score_loss(y, p), "base_rate": y.mean()}


logit = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), CATEGORICAL),
    ])),
    ("model", LogisticRegression(max_iter=2000)),
])
logit.fit(X_train, y_train)

results = [evaluate("logistic (valid)", y_valid, logit.predict_proba(X_valid)[:, 1])]
pd.DataFrame(results).round(4)

## 3. XGBoost

Categoricals are ordinal-encoded (trees split on them fine) and numeric missing values stay as `NaN`,
which XGBoost handles natively. Early stopping on the 2021 validation year picks the number of trees.

In [ ]:
from xgboost import XGBClassifier

prep = ColumnTransformer([
    ("num", "passthrough", NUMERIC),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL),
])
prep.fit(X_train)

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="aucpr",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb.fit(prep.transform(X_train), y_train,
        eval_set=[(prep.transform(X_valid), y_valid)], verbose=100)

# One object that goes from raw feature columns to probabilities: this is what gets deployed.
xgb_pipe = Pipeline([("prep", prep), ("model", xgb)])

results.append(evaluate("xgboost (valid)", y_valid, xgb_pipe.predict_proba(X_valid)[:, 1]))
pd.DataFrame(results).round(4)

## 4. Final check on the untouched test year

In [ ]:
p_test_logit = logit.predict_proba(X_test)[:, 1]
p_test_xgb = xgb_pipe.predict_proba(X_test)[:, 1]
test_metrics = pd.DataFrame([
    evaluate("logistic (test)", y_test, p_test_logit),
    evaluate("xgboost (test)", y_test, p_test_xgb),
]).round(4)
test_metrics

In [ ]:
# Lift by risk decile: does the top 10% of scores capture a big share of litigated claims?
lift = pd.DataFrame({"p": p_test_xgb, "y": y_test.values})
lift["decile"] = pd.qcut(lift.p.rank(method="first", ascending=False), 10, labels=range(1, 11))
lift_table = lift.groupby("decile", observed=True).agg(n=("y", "size"), avg_pred=("p", "mean"), actual_rate=("y", "mean"))
lift_table["lift"] = lift_table.actual_rate / lift.y.mean()
lift_table["cum_share_of_litigated"] = lift.groupby("decile", observed=True).y.sum().cumsum() / lift.y.sum()
lift_table.round(3)

In [ ]:
# Which features the trees used most (gain-based importance)
pd.Series(xgb.get_booster().get_score(importance_type="gain")) \
  .rename(index=lambda f: FEATURES[int(f[1:])]) \
  .sort_values(ascending=False).round(1).to_frame("gain")

## 5. Register the model in Snowflake

The Model Registry stores the pipeline, its Python dependencies, an input/output signature and metrics.
Once registered, the model can be called from Python (`mv.run`) **or from plain SQL**, which is what
`deploy/04_deploy_scoring.sql` uses.

In [ ]:
from datetime import datetime

import sklearn
import xgboost
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="WC_ML", schema_name="MODELS")
version = "V_" + datetime.now().strftime("%Y%m%d_%H%M%S")
xgb_test = test_metrics.set_index("model").loc["xgboost (test)"]

mv = reg.log_model(
    xgb_pipe,
    model_name=MODEL_NAME,
    version_name=version,
    sample_input_data=X_train.head(500),
    # pin the versions used for training so the pickled pipeline loads identically
    conda_dependencies=[f"scikit-learn=={sklearn.__version__}", f"xgboost=={xgboost.__version__}"],
    metrics={
        "test_year": TEST_YEAR,
        "test_roc_auc": float(xgb_test.roc_auc),
        "test_pr_auc": float(xgb_test.pr_auc),
        "test_brier": float(xgb_test.brier),
        "features": FEATURES,
    },
    comment="Probability a NY workers' comp claim gets attorney representation, using intake-time features.",
)
reg.get_model(MODEL_NAME).default = version
print("Registered", MODEL_NAME, version)

In [ ]:
# The functions the model exposes, with their argument order and output names
mv.show_functions()

In [ ]:
# Smoke test: run the registered model inside Snowflake on a few 2023 claims (never seen in training)
new_claims = session.table(FEATURE_VIEW).filter(F.col("ASSEMBLY_YEAR") == 2023).limit(10)
mv.run(new_claims, function_name="predict_proba").to_pandas()

In [ ]:
reg.show_models()

**Next:** open `deploy/04_deploy_scoring.sql` in a SQL worksheet to call the model from SQL, create the batch
scoring procedure, and schedule it with a Task.